In [1]:
import os
import pandas as pd
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'/Users/ryan/github/prosodic')
# !pip install -r /Users/ryan/github/prosodic/requirements.txt

import prosodic

# from collections import defaultdict
# import random

# def count_syllables(text):
#     """Count syllables using prosodic"""
#     try:
#         return prosodic.Text(txt=text).num_sylls
#     except:
#         return 0

# def build_markov_chain(texts, n=2):
#     """Build a Markov chain from a list of texts with line breaks preserved"""
#     chain = defaultdict(list)
    
#     for text in texts:
#         # Process each line separately
#         for line in text.split('\n'):
#             if not line.strip(): continue
#             words = ['<START>'] + line.split() + ['<END>']
            
#             for i in range(len(words) - n):
#                 state = tuple(words[i:i+n])
#                 next_word = words[i+n]
#                 chain[state].append(next_word)
    
#     return chain

# def generate_line(chain, n=2, target_sylls=10, max_attempts=50):
#     """Generate a single line with target syllables"""
#     current = None
#     for _ in range(max_attempts):
#         current = ('<START>',) * (n-1) + ('<START>',)
#         line_words = []
        
#         while True:
#             if current not in chain or not chain[current]:
#                 break
                
#             next_word = random.choice(chain[current])
#             if next_word == '<END>':
#                 break
                
#             test_line = ' '.join(line_words + [next_word])
#             sylls = count_syllables(test_line)
            
#             if sylls > target_sylls:
#                 break
#             elif sylls == target_sylls:
#                 line_words.append(next_word)
#                 return ' '.join(line_words)
            
#             line_words.append(next_word)
#             current = tuple(list(current[1:]) + [next_word])
    
#     return current

# def generate_sonnet(chain, n=2):
#     """Generate a 14-line sonnet with 10 syllables per line"""
#     lines = []
    
#     while len(lines) < 14:
#         line = generate_line(chain, n)
#         print(line)
#         if line:
#             lines.append(line)
    
#     return '\n'.join(lines)

# # Train the model
# shaksonnets = prosodic.Text(fn='/Users/ryan/github/prosodic/corpora/corppoetry_en/en.shakespeare.txt')
# poems = [stanza.txt.strip() for stanza in shaksonnets.stanzas]
# markov_chain = build_markov_chain(poems, n=2)

# # Generate a sonnet
# generated_sonnet = generate_sonnet(markov_chain)
# print(generated_sonnet)

In [2]:

!pip install markovify



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import markovify

# Get raw text as string
with open('/Users/ryan/github/prosodic/corpora/corppoetry_en/en.shakespeare.txt') as f:
    text = f.read()

# Build the model
text_model = markovify.Text(text)

@prosodic.cache
def get_shakespeare_markov():
    with open('/Users/ryan/github/prosodic/corpora/corppoetry_en/en.shakespeare.txt') as f:
        text = f.read()
    return markovify.Text(text)

@prosodic.cache
def get_human_markov():
    df=pd.read_pickle('data.allpoems.pkl')

    def filter_poem_txt(txt):
        vparas = [para.strip() for para in txt.split('\n\n') if '\n' in para.strip()]
        return '\n\n'.join(vparas)

    def num_lines_txt(txt):
        return len([v for v in txt.splitlines() if v.strip()])

    df['poem_txt'] = df['poem'].apply(filter_poem_txt)
    df['num_lines_txt'] = df['poem_txt'].apply(num_lines_txt)

    df['prompt'] = df['prompt'].fillna('')
    # df_sonnets = df[df.prompt.str.contains('sonnet') | (df.prompt == '')]
    df = df[df.model.str.startswith('b.')]
    df_sonnets_smpl = pd.concat(gdf.sample(frac=1).iloc[:154] for _,gdf in df.groupby('model'))
    df_sonnets = df_sonnets_smpl[df_sonnets_smpl.num_lines==14]
    df_sonnets.model.value_counts()
    text = '\n\n'.join(df_sonnets['poem_txt'])
    return markovify.Text(text)



# get_shakespeare_markov()
get_human_markov()

In [4]:
def lines_from_sentence(sentence):
    lines = []
    line_words = []
    failed = False
    for word in sentence.split():
        line_words.append(word)
        try:
            line = prosodic.Text(' '.join(line_words)).line1
            line_num_sylls = line.num_sylls
        except Exception as e:
            failed = True
            break
        if line_num_sylls != 10:
            continue
        else:
            lines.append(line)
            line_words = []
    return lines if not line_words and not failed else []

def lines_from_markov(text_model=None):
    if text_model is None:
        text_model = get_shakespeare_markov()
    lines = []
    while not lines:
        sent = text_model.make_sentence()
        if sent:
            lines = lines_from_sentence(sent)
    return lines

# lines_from_markov()

def sonnet_from_markov(run=0, text_model=None):
    lines = []
    
    while len(lines) < 14:
        lines.extend(lines_from_markov(text_model=text_model))
        if len(lines) > 14:
            lines = lines[:14]
            # print('retrying')
            # lines = []
    return '\n'.join(line.txt.strip() for line in lines)

def shakespeare_sonnet_from_markov(run=0):
    return sonnet_from_markov(run, text_model=get_shakespeare_markov())

def human_sonnet_from_markov(run=0):
    return sonnet_from_markov(run, text_model=get_human_markov())


print(human_sonnet_from_markov())

As when upon our winter-blighted
lea There comes the breath drawn at your touch:
I no longer believe what I hear.
Sweet were the joys, which once you did
possess, When on the watch to trap and
snare The incautious hearts of all my
thinking trembles into
nought And all the young and fair, And through
the ages groan A note of misery.
They to despair themselves Alley, Who
with their backward view, All they behold.
It was not as if the jasmine scent
And the approaching Sun when he can?
Fair Mistress of the sheak trees forlorn to


In [8]:
markov_stash = prosodic.HashStash('markov_sonnets')

markov_res = markov_stash.map(sonnet_from_markov, objects = [(i,) for i in range(154*10)], num_proc=1)
l = markov_res.results
len(l)

[9.18s] Mapping __main__.sonnet_from_markov across 1540 objects:  37%|███▋      | 563/1540 [1:00:17<7:54:49, 29.16s/it]

In [6]:
# human_markov_res = markov_stash.map(human_sonnet_from_markov, objects = [(i,) for i in range(154*5)], num_proc=2)
# l2 = human_markov_res.results
# len(l2)


308

In [7]:
outld = [{'model': 'shakespeare-markov', 'poem_txt': l[i], 'num_lines_txt': len(l[i].splitlines())} for i in range(len(l))]
outld += [{'model': 'human-markov', 'poem_txt': l2[i], 'num_lines_txt': len(l2[i].splitlines())} for i in range(len(l2))]
outdf = pd.DataFrame(outld)
outdf.to_pickle('data.markov_sonnets.pkl')
outdf

,model,poem_txt,num_lines_txt
0,shakespeare-markov,You are my all the all of me.\nSpend'st thou t...,14
1,shakespeare-markov,"Tired with all hearts, Which I by\nyours, you'...",14
2,shakespeare-markov,"The summer's flower is to render thee.\nO, cal...",14
3,shakespeare-markov,"When forty winters shall beseige\nthy brow, An...",14
4,shakespeare-markov,But do thy worst all best exceeds?\nIf that be...,14
...,...,...,...
611,human-markov,"What know we of the Songs of the tide,\nBroken...",14
612,human-markov,Or as my love a calmer tribute pays!\nThen I s...,14
613,human-markov,Martha is not a whit the wiser.\nWalking throu...,14
614,human-markov,"Bird of the summer, lovely summer's pride,\nSw...",14
